# [5.6] Embedding Retrieval and Function-Calling Controls - Solutions

This notebook runs the solved local contracts and then displays the report-backed signature result. Keep the claim boundary in view: this is text retrieval and function-calling evidence, not a broad multimodal/VLM interpretability section.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter5_modern_architectures"
section = "part6_multimodal_embedding_function_models"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_multimodal_embedding_function_models.tests as tests
import part6_multimodal_embedding_function_models.solutions as solutions

mean_pool_embeddings = solutions.mean_pool_embeddings
cosine_similarity_matrix = solutions.cosine_similarity_matrix
retrieval_ranks = solutions.retrieval_ranks
embedding_retrieval_report = solutions.embedding_retrieval_report
fit_centroid_probe = solutions.fit_centroid_probe
predict_centroid_probe = solutions.predict_centroid_probe
centroid_probe_accuracy = solutions.centroid_probe_accuracy
mask_disallowed_tools = solutions.mask_disallowed_tools
function_call_report = solutions.function_call_report
parse_function_call_text = solutions.parse_function_call_text
schema_token_attribution = solutions.schema_token_attribution

## Embedding Geometry

<details><summary>Expected output</summary>

```text
All tests in `test_mean_pool_embeddings_ignores_padding_and_matches_reference` passed!
All tests in `test_retrieval_metrics_rank_pairs_and_hard_negative_margin` passed!
All tests in `test_centroid_probe_recovers_heldout_clusters` passed!
```

</details>

<details><summary>Help - what these tests isolate</summary>

Pooling removes padding artifacts; retrieval checks paired ranks and margins; the centroid probe checks whether a simple concept geometry survives held-out points.

</details>

In [ ]:
tests.test_mean_pool_embeddings_ignores_padding_and_matches_reference(mean_pool_embeddings)
tests.test_retrieval_metrics_rank_pairs_and_hard_negative_margin(
    cosine_similarity_matrix,
    retrieval_ranks,
    embedding_retrieval_report,
)
tests.test_centroid_probe_recovers_heldout_clusters(
    fit_centroid_probe,
    predict_centroid_probe,
    centroid_probe_accuracy,
)

## Tool-Use Contracts

<details><summary>Expected output</summary>

```text
All tests in `test_mask_disallowed_tools_blocks_invalid_logits` passed!
All tests in `test_function_call_report_separates_tool_and_abstention_errors` passed!
All tests in `test_parse_function_call_text_extracts_name_and_arguments` passed!
All tests in `test_schema_token_attribution_matches_dot_products` passed!
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Help - why the parser and controls come first</summary>

Before interpreting a tool model, you need to know whether the tool was allowed, whether no-call prompts were evaluated, whether the text span parsed, and whether schema attribution is being treated as a hypothesis rather than causal proof.

</details>

In [ ]:
tests.test_mask_disallowed_tools_blocks_invalid_logits(mask_disallowed_tools)
tests.test_function_call_report_separates_tool_and_abstention_errors(function_call_report)
tests.test_parse_function_call_text_extracts_name_and_arguments(parse_function_call_text)
tests.test_schema_token_attribution_matches_dot_products(schema_token_attribution)
tests.test_notebook_contract(solutions.run_smoke_test)

## Signature Result

<details><summary>Interpreting the signature result</summary>

The retrieval controls are clean: both embedding models retrieve the paired documents and fail the permuted pairing. FunctionGemma's parser and tool names are perfect on the deterministic slice, but argument matching is imperfect. That imperfection is the point students should inspect.

</details>

In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test()
summary = {
    "bge": {
        "top1": gpu["bge_retrieval_top1_accuracy"],
        "margin": round(gpu["bge_mean_margin"], 3),
        "permuted_top1": gpu["bge_permuted_top1_accuracy"],
    },
    "embeddinggemma": {
        "top1": gpu["embeddinggemma_retrieval_top1_accuracy"],
        "margin": round(gpu["embeddinggemma_mean_margin"], 3),
        "permuted_top1": gpu["embeddinggemma_permuted_top1_accuracy"],
    },
    "functiongemma": {
        "parse_accuracy": gpu["functiongemma_parse_accuracy"],
        "function_name_accuracy": gpu["functiongemma_function_name_accuracy"],
        "exact_argument_accuracy": gpu["functiongemma_exact_argument_accuracy"],
        "required_argument_accuracy": gpu["functiongemma_required_argument_accuracy"],
        "failure_indices": gpu["functiongemma_failure_indices"],
    },
    "peak_vram_gb": round(gpu["peak_vram_gb"], 3),
}
summary

## Limitations

The report proves scoped retrieval and function-calling checks. It does not prove broad multimodal modeling, image-token flow, causal schema mechanisms, or perfect tool-use behavior.